# Post-Orthodontic Smile Video Simulation

In [ ]:
import os
import cv2
import glob
import base64
import time
import requests
import json
import urllib
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display

# Define call rules

Please modify the following code blocks based on the credentials you obtained from us.

In [ ]:
# Chohotech service request URL, sent with the API documentation.
base_url = "<service request URL>"

# Chohotech file service URL, sent with the API documentation.
file_server_url = "<service file server URL>"

# The authentication header must be passed in. Please keep the TOKEN confidential!!! If it is leaked, please contact us immediately to reset it. All tasks using this TOKEN will be charged to your account.
zh_token = "<your company's service Token, sent with the contact>" # All API calls must be authenticated with the token.

user_group = "APIClient" # User group, usually named APIClient.

# Your company's user_id, sent with the API documentation.
user_id = "<your company's user_id>"

# If you have received creds.json, it will be read directly below.
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

## file upload 

In [ ]:
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # Must specify postfix, i.e., the file extension
                        headers={"X-ZH-TOKEN": zh_token}) # Get the signed upload URL
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # Returns a single string JSON "string", can also use json.loads(resp.text)

    resp = requests.put(upload_url, data) # No auth header is needed for uploading to the cloud storage service

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
      "Content-Type": "application/json",
      "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)
    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")

    print("API finished in {}s".format(time.time()-start_time))
    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    return response.json()

def show_video(urn, output_path="result.mp4", show_info=True):
    # Download the video in binary format
    resp = requests.get(
        file_server_url + "/file/download",
        params={"urn": urn},
        headers={"X-ZH-TOKEN": zh_token},
        timeout=60
    )
    resp.raise_for_status()

    # Save as a local video file
    with open(output_path, "wb") as f:
        f.write(resp.content)

    print(f"Video saved to: {output_path}")

    if not show_info:
        return

    # Retrieve video metadata with OpenCV
    cap = cv2.VideoCapture(output_path)
    if not cap.isOpened():
        print("Warning: video saved but cannot be opened by OpenCV")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0

    cap.release()

    print(
        f"Video info:\n"
        f"  Resolution : {width} x {height}\n"
        f"  FPS        : {fps:.2f}\n"
        f"  Frames     : {frame_count}\n"
        f"  Duration   : {duration:.2f} sec"
    )


## Smile-Video-Simulation

Please refer to: https://www.chohotech.com/docs/cloud-en/#/module/smile-video-simulation-1

In [ ]:
json_call = {
  "spec_group": "smile",
  "spec_name": "smile-video-simulation",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {"video": upload_file("smile.mp4")}
}
result = run_job_and_get_results(json_call, 100)

In [ ]:
print(result)

### Visualization

In [ ]:
show_video(result['result']['video'])

In [ ]:
display(Video("result.mp4", embed=True))